## Apartado 2: Transformaciones y modelado en Silver

**Objetivo**: enriquecer los vuelos con aerolínea y ciudades, crear métricas por vuelo, agregar por aerolínea y aeropuerto, validar y guardar en Silver.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

CATALOG = "aerolinea_trabajo_final"

def guardar_delta(df, schema, tabla):
    nombre = f"{CATALOG}.{schema}.{tabla}"
    (df.write
       .format("delta")
       .mode("overwrite")
       .option("overwriteSchema", "true")
       .saveAsTable(nombre))
    print(f"Guardada {nombre}: {spark.table(nombre).count()} filas")

#### 2.1 Cargar las tablas Bronze y revisar esquemas

In [0]:
flights_b = spark.table(f"{CATALOG}.bronze.flights")
airlines_b = spark.table(f"{CATALOG}.bronze.airlines")
airports_b = spark.table(f"{CATALOG}.bronze.airports")

for nombre, df in [("flights", flights_b), ("airlines", airlines_b), ("airports", airports_b)]:
    print(f"--- {nombre}: {df.count()} filas")
    df.printSchema()

--- flights: 49479 filas
root
 |-- flight_id: string (nullable = true)
 |-- flight_date: date (nullable = true)
 |-- airline_id: string (nullable = true)
 |-- origin_airport: string (nullable = true)
 |-- destination_airport: string (nullable = true)
 |-- dep_delay: integer (nullable = true)
 |-- arr_delay: integer (nullable = true)
 |-- distance_miles: integer (nullable = true)
 |-- source_file: string (nullable = true)
 |-- ingest_timestamp: timestamp (nullable = true)

--- airlines: 6 filas
root
 |-- airline_id: string (nullable = true)
 |-- airline_name: string (nullable = true)
 |-- source_file: string (nullable = true)
 |-- ingest_timestamp: timestamp (nullable = true)

--- airports: 10 filas
root
 |-- airport_id: string (nullable = true)
 |-- city: string (nullable = true)
 |-- country: string (nullable = true)
 |-- source_file: string (nullable = true)
 |-- ingest_timestamp: timestamp (nullable = true)



#### 2.2 Unir tablas: nombre de aerolínea y ciudad/país de origen y destino

In [0]:
origen = airports_b.select(F.col("airport_id").alias("origin_airport"),
                           F.col("city").alias("origin_city"),
                           F.col("country").alias("origin_country"))
destino = airports_b.select(F.col("airport_id").alias("destination_airport"),
                            F.col("city").alias("destination_city"),
                            F.col("country").alias("destination_country"))

flights_joined = (flights_b.drop("ingest_timestamp", "source_file")
    .join(F.broadcast(airlines_b.select("airline_id", "airline_name")), "airline_id", "left")
    .join(F.broadcast(origen), "origin_airport", "left")
    .join(F.broadcast(destino), "destination_airport", "left"))

display(flights_joined.limit(10))

destination_airport,origin_airport,airline_id,flight_id,flight_date,dep_delay,arr_delay,distance_miles,airline_name,origin_city,origin_country,destination_city,destination_country
DEN,SEA,WN,FL-100022,2025-01-31,-25,-27,2468,Southwest Airlines,Seattle,USA,Denver,USA
DFW,SFO,DL,FL-100039,2025-02-10,7,-8,2366,Delta Air Lines,San Francisco,USA,Dallas,USA
DFW,ORD,NK,FL-100068,2025-01-17,-1,1,455,Spirit Airlines,Chicago,USA,Dallas,USA
ORD,MIA,NK,FL-100072,2025-04-30,20,33,1813,Spirit Airlines,Miami,USA,Chicago,USA
JFK,LAS,B6,FL-100073,2025-01-22,-10,-18,405,JetBlue Airways,Las Vegas,USA,New York,USA
ATL,LAS,B6,FL-100097,2025-05-14,18,27,2610,JetBlue Airways,Las Vegas,USA,Atlanta,USA
SEA,LAS,DL,FL-100115,2025-02-04,-10,-2,1899,Delta Air Lines,Las Vegas,USA,Seattle,USA
DFW,ATL,UA,FL-100118,2025-05-27,-36,-29,1503,United Airlines,Atlanta,USA,Dallas,USA
LAX,JFK,UA,FL-100132,2025-06-05,0,-8,1142,United Airlines,New York,USA,Los Angeles,USA
LAS,LAX,DL,FL-100147,2025-02-23,12,10,804,Delta Air Lines,Los Angeles,USA,Las Vegas,USA


#### 2.3 Crear nuevas columnas

| Columna | Regla |
|---|---|
| `delay_total` | `dep_delay + arr_delay` |
| `on_time` | `true` si el retraso de llegada es ≤ 15 minutos (criterio estándar de la industria) |
| `distance_km` | `distance_miles × 1.60934` |
| `delay_category` | Según el retraso de llegada: **None** ≤ 0 min · **Short** 1–15 · **Medium** 16–60 · **Long** > 60 |

In [0]:
def agregar_columnas_derivadas(df):
    arr = F.col("arr_delay")
    return (df
        .withColumn("delay_total", F.col("dep_delay") + F.col("arr_delay"))
        .withColumn("on_time", F.when(arr.isNull(), F.lit(None)).otherwise(arr <= 15))
        .withColumn("distance_km", F.round(F.col("distance_miles") * F.lit(1.60934), 2))
        .withColumn("delay_category",
                    F.when(arr.isNull(), "Unknown")
                     .when(arr <= 0, "None")
                     .when(arr <= 15, "Short")
                     .when(arr <= 60, "Medium")
                     .otherwise("Long")))

flights_enriched = agregar_columnas_derivadas(flights_joined)
display(flights_enriched.groupBy("delay_category").count().orderBy("delay_category"))

delay_category,count
Long,1013
Medium,21645
None,14118
Short,12468
Unknown,235


#### 2.4 Agregaciones por aerolínea

In [0]:
def metricas_aerolinea(df):
    return (df.groupBy("airline_id", "airline_name")
              .agg(F.round(F.avg("dep_delay"), 2).alias("avg_dep_delay"),
                   F.round(F.avg("arr_delay"), 2).alias("avg_arr_delay"),
                   F.round(F.avg("delay_total"), 2).alias("avg_delay_total"),
                   F.round(F.sum("distance_km"), 2).alias("total_distance_km"),
                   F.count("*").alias("num_flights"),
                   F.round(F.avg(F.col("on_time").cast("int")) * 100, 2).alias("pct_on_time"))
              .orderBy("airline_id"))

def puntualidad_aerolinea_por_origen(df):
    return (df.groupBy("airline_id", "airline_name", "origin_airport", "origin_city")
              .agg(F.count("*").alias("num_flights"),
                   F.round(F.avg(F.col("on_time").cast("int")) * 100, 2).alias("pct_on_time"))
              .orderBy("airline_id", "origin_airport"))

display(metricas_aerolinea(flights_enriched))
display(puntualidad_aerolinea_por_origen(flights_enriched))

airline_id,airline_name,avg_dep_delay,avg_arr_delay,avg_delay_total,total_distance_km,num_flights,pct_on_time
AA,American Airlines,4.99,9.86,14.85,2.034555008E7,8431,59.78
B6,JetBlue Airways,14.86,19.85,34.69,2.009222087E7,8227,42.42
DL,Delta Air Lines,5.26,10.11,15.42,2.008208165E7,8292,59.98
NK,Spirit Airlines,14.85,19.9,34.71,1.993559489E7,8206,42.76
UA,United Airlines,5.4,10.32,15.71,1.936626337E7,8075,60.06
WN,Southwest Airlines,5.06,10.23,15.29,2.006173048E7,8248,58.84


airline_id,airline_name,origin_airport,origin_city,num_flights,pct_on_time
AA,American Airlines,ATL,Atlanta,880,60.5
AA,American Airlines,DEN,Denver,805,58.7
AA,American Airlines,DFW,Dallas,857,60.07
AA,American Airlines,JFK,New York,821,61.34
AA,American Airlines,LAS,Las Vegas,833,58.45
AA,American Airlines,LAX,Los Angeles,821,62.22
AA,American Airlines,MIA,Miami,842,58.16
AA,American Airlines,ORD,Chicago,908,58.78
AA,American Airlines,SEA,Seattle,835,58.53
AA,American Airlines,SFO,San Francisco,829,61.09


#### 2.5 Agregaciones por aeropuerto

In [0]:
def metricas_aeropuerto(df, aeropuertos):
    salidas = (df.groupBy(F.col("origin_airport").alias("airport_id"))
                 .agg(F.count("*").alias("num_departures"),
                      F.round(F.avg("dep_delay"), 2).alias("avg_dep_delay_as_origin"),
                      F.round(F.avg(F.col("on_time").cast("int")) * 100, 2).alias("pct_on_time_as_origin")))
    llegadas = (df.groupBy(F.col("destination_airport").alias("airport_id"))
                  .agg(F.count("*").alias("num_arrivals"),
                       F.round(F.avg("arr_delay"), 2).alias("avg_arr_delay_as_destination"),
                       F.round(F.avg(F.col("on_time").cast("int")) * 100, 2).alias("pct_on_time_as_destination")))
    return (aeropuertos.select("airport_id", "city", "country")
              .join(salidas, "airport_id", "left")
              .join(llegadas, "airport_id", "left")
              .withColumn("total_flights", F.coalesce("num_departures", F.lit(0)) + F.coalesce("num_arrivals", F.lit(0)))
              .orderBy("airport_id"))

display(metricas_aeropuerto(flights_enriched, airports_b))

airport_id,city,country,num_departures,avg_dep_delay_as_origin,pct_on_time_as_origin,num_arrivals,avg_arr_delay_as_destination,pct_on_time_as_destination,total_flights
ATL,Atlanta,USA,5010,8.14,54.73,4869,13.04,54.61,9879
DEN,Denver,USA,5023,8.06,54.43,5046,13.67,53.43,10069
DFW,Dallas,USA,4915,8.85,54.1,4945,13.62,53.62,9860
JFK,New York,USA,4899,8.12,54.03,4891,13.4,54.59,9790
LAS,Las Vegas,USA,4962,8.82,53.12,5024,13.7,53.21,9986
LAX,Los Angeles,USA,4965,8.38,54.02,4976,13.62,53.74,9941
MIA,Miami,USA,4878,8.3,53.71,4852,13.0,54.57,9730
ORD,Chicago,USA,5007,8.52,53.56,5031,13.2,54.21,10038
SEA,Seattle,USA,4891,8.33,53.89,4826,12.48,54.94,9717
SFO,San Francisco,USA,4929,8.39,54.29,5019,13.89,53.05,9948


#### 2.6 Validación y limpieza final

1. Se revisan los nulos en las columnas importantes.
2. Los retrasos faltantes se imputan con la **mediana de la aerolínea** y se marca la fila con `delay_imputed = true`, para no perder el vuelo en los conteos y poder excluirlo luego del modelo de ML.
3. Los textos faltantes de los joins se rellenan con `"Unknown"`.
4. Se recalculan las columnas derivadas con los valores ya completos.

In [0]:
columnas_clave = ["flight_id", "flight_date", "airline_id", "airline_name", "origin_airport",
                  "destination_airport", "origin_city", "destination_city", "dep_delay",
                  "arr_delay", "distance_miles"]
print("Nulos ANTES de la limpieza final:")
display(flights_enriched.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in columnas_clave]))

Nulos ANTES de la limpieza final:


flight_id,flight_date,airline_id,airline_name,origin_airport,destination_airport,origin_city,destination_city,dep_delay,arr_delay,distance_miles
0,0,0,0,0,0,0,0,239,235,0


In [0]:
medianas = (flights_joined.groupBy("airline_id")
            .agg(F.percentile_approx("dep_delay", 0.5).alias("med_dep"),
                 F.percentile_approx("arr_delay", 0.5).alias("med_arr")))

flights_validated = (flights_joined
    .join(F.broadcast(medianas), "airline_id", "left")
    .withColumn("delay_imputed", F.col("dep_delay").isNull() | F.col("arr_delay").isNull())
    .withColumn("dep_delay", F.coalesce("dep_delay", "med_dep"))
    .withColumn("arr_delay", F.coalesce("arr_delay", "med_arr"))
    .drop("med_dep", "med_arr")
    .na.fill("Unknown", subset=["airline_name", "origin_city", "origin_country",
                                "destination_city", "destination_country"]))

flights_silver = (agregar_columnas_derivadas(flights_validated)
    .withColumn("silver_timestamp", F.current_timestamp())
    .select("flight_id", "flight_date", "airline_id", "airline_name",
            "origin_airport", "origin_city", "origin_country",
            "destination_airport", "destination_city", "destination_country",
            "dep_delay", "arr_delay", "delay_total", "on_time", "delay_category",
            "distance_miles", "distance_km", "delay_imputed", "silver_timestamp"))

print("Nulos DESPUÉS de la limpieza final:")
display(flights_silver.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in columnas_clave]))
print("Filas con retrasos imputados:", flights_silver.filter("delay_imputed").count())

Nulos DESPUÉS de la limpieza final:


flight_id,flight_date,airline_id,airline_name,origin_airport,destination_airport,origin_city,destination_city,dep_delay,arr_delay,distance_miles
0,0,0,0,0,0,0,0,0,0,0


Filas con retrasos imputados: 473


#### 2.7 Guardar en Silver

In [0]:
guardar_delta(flights_silver, "silver", "flights_enriched")
guardar_delta(metricas_aerolinea(flights_silver), "silver", "airline_metrics")
guardar_delta(puntualidad_aerolinea_por_origen(flights_silver), "silver", "airline_origin_ontime")
guardar_delta(metricas_aeropuerto(flights_silver, airports_b), "silver", "airport_metrics")

Guardada aerolinea_trabajo_final.silver.flights_enriched: 49479 filas
Guardada aerolinea_trabajo_final.silver.airline_metrics: 6 filas
Guardada aerolinea_trabajo_final.silver.airline_origin_ontime: 60 filas
Guardada aerolinea_trabajo_final.silver.airport_metrics: 10 filas


## Apartado 3: Análisis avanzado y métricas

**Objetivo**: consultas complejas y funciones de ventana sobre los vuelos de Silver.

#### 3.1 Análisis temporal de retrasos (mes, día de la semana y trimestre)

In [0]:
flights_s = spark.table(f"{CATALOG}.silver.flights_enriched")

flights_time = (flights_s
    .withColumn("month", F.month("flight_date"))
    .withColumn("month_name", F.date_format("flight_date", "MMMM"))
    .withColumn("day_of_week", F.dayofweek("flight_date"))          # 1 = domingo ... 7 = sábado
    .withColumn("day_name", F.date_format("flight_date", "EEEE"))
    .withColumn("quarter", F.quarter("flight_date")))

def promedio_retraso(df, *cols):
    return (df.groupBy(*cols)
              .agg(F.round(F.avg("dep_delay"), 2).alias("avg_dep_delay"),
                   F.round(F.avg("arr_delay"), 2).alias("avg_arr_delay"),
                   F.round(F.avg("delay_total"), 2).alias("avg_delay_total"),
                   F.count("*").alias("num_flights"))
              .orderBy(*cols))

delay_by_month = promedio_retraso(flights_time, "month", "month_name")
delay_by_weekday = promedio_retraso(flights_time, "day_of_week", "day_name")
delay_by_quarter = promedio_retraso(flights_time, "quarter")

display(delay_by_month)
display(delay_by_weekday)
display(delay_by_quarter)

month,month_name,avg_dep_delay,avg_arr_delay,avg_delay_total,num_flights
1,January,8.7,13.72,22.42,8577
2,February,8.13,13.01,21.13,7640
3,March,8.14,12.97,21.1,8421
4,April,8.39,13.51,21.9,8109
5,May,8.18,13.22,21.4,8401
6,June,8.79,13.73,22.52,8331


day_of_week,day_name,avg_dep_delay,avg_arr_delay,avg_delay_total,num_flights
1,Sunday,8.35,13.46,21.81,7193
2,Monday,8.63,13.67,22.3,7122
3,Tuesday,8.55,13.42,21.97,6803
4,Wednesday,8.68,13.68,22.36,7131
5,Thursday,8.4,13.43,21.83,7143
6,Friday,7.78,12.67,20.45,7119
7,Saturday,8.35,13.22,21.57,6968


quarter,avg_dep_delay,avg_arr_delay,avg_delay_total,num_flights
1,8.33,13.24,21.57,24638
2,8.45,13.49,21.94,24841


#### 3.2 Top 5 vuelos con mayor retraso por aerolínea: 
`row_number` vs `dense_rank`

- `row_number()` numera 1, 2, 3… sin repetir: ante empates elige uno arbitrariamente, así que siempre devuelve **exactamente 5** vuelos por aerolínea.
- `dense_rank()` da el **mismo puesto a los empates** y no deja huecos: el "top 5" puede contener **más de 5 vuelos** si varios comparten retraso.

In [0]:
w_airline = Window.partitionBy("airline_id").orderBy(F.col("delay_total").desc())

ranked = (flights_s
    .withColumn("row_number", F.row_number().over(w_airline))
    .withColumn("dense_rank", F.dense_rank().over(w_airline)))

cols_top = ["airline_id", "airline_name", "flight_id", "flight_date", "origin_airport",
            "destination_airport", "dep_delay", "arr_delay", "delay_total", "row_number", "dense_rank"]

top5_row_number = ranked.filter("row_number <= 5").select(cols_top).orderBy("airline_id", "row_number")
top5_dense_rank = ranked.filter("dense_rank <= 5").select(cols_top).orderBy("airline_id", "dense_rank", "flight_id")

display(top5_row_number)
display(top5_dense_rank)

airline_id,airline_name,flight_id,flight_date,origin_airport,destination_airport,dep_delay,arr_delay,delay_total,row_number,dense_rank
AA,American Airlines,FL-114276,2025-06-29,SEA,LAS,79,85,164,1,1
AA,American Airlines,FL-106973,2025-06-05,DEN,SEA,71,83,154,2,2
AA,American Airlines,FL-125098,2025-03-31,ORD,DEN,73,81,154,3,2
AA,American Airlines,FL-148219,2025-06-16,SFO,ORD,70,79,149,4,3
AA,American Airlines,FL-128580,2025-03-24,DEN,SEA,74,71,145,5,4
B6,JetBlue Airways,FL-139444,2025-01-14,SFO,DEN,87,112,199,1,1
B6,JetBlue Airways,FL-121487,2025-01-23,SFO,ATL,78,101,179,2,2
B6,JetBlue Airways,FL-122541,2025-03-14,SFO,MIA,74,104,178,3,3
B6,JetBlue Airways,FL-122207,2025-05-19,DEN,ATL,83,92,175,4,4
B6,JetBlue Airways,FL-100239,2025-02-10,MIA,ATL,77,93,170,5,5


airline_id,airline_name,flight_id,flight_date,origin_airport,destination_airport,dep_delay,arr_delay,delay_total,row_number,dense_rank
AA,American Airlines,FL-114276,2025-06-29,SEA,LAS,79,85,164,1,1
AA,American Airlines,FL-106973,2025-06-05,DEN,SEA,71,83,154,2,2
AA,American Airlines,FL-125098,2025-03-31,ORD,DEN,73,81,154,3,2
AA,American Airlines,FL-148219,2025-06-16,SFO,ORD,70,79,149,4,3
AA,American Airlines,FL-128580,2025-03-24,DEN,SEA,74,71,145,5,4
AA,American Airlines,FL-128036,2025-03-30,ORD,LAS,70,72,142,6,5
B6,JetBlue Airways,FL-139444,2025-01-14,SFO,DEN,87,112,199,1,1
B6,JetBlue Airways,FL-121487,2025-01-23,SFO,ATL,78,101,179,2,2
B6,JetBlue Airways,FL-122541,2025-03-14,SFO,MIA,74,104,178,3,3
B6,JetBlue Airways,FL-122207,2025-05-19,DEN,ATL,83,92,175,4,4


In [0]:
# Comparación: cuántos vuelos devuelve cada método por aerolínea
comparacion = (top5_row_number.groupBy("airline_id").agg(F.count("*").alias("vuelos_row_number"))
    .join(top5_dense_rank.groupBy("airline_id").agg(F.count("*").alias("vuelos_dense_rank")), "airline_id")
    .withColumn("diferencia", F.col("vuelos_dense_rank") - F.col("vuelos_row_number"))
    .orderBy("airline_id"))
display(comparacion)

# Filas donde ambos rankings difieren (evidencia de empates)
display(ranked.filter("dense_rank <= 5 AND row_number <> dense_rank").select(cols_top).orderBy("airline_id", "row_number"))

airline_id,vuelos_row_number,vuelos_dense_rank,diferencia
AA,5,6,1
B6,5,6,1
DL,5,6,1
NK,5,7,2
UA,5,6,1
WN,5,5,0


airline_id,airline_name,flight_id,flight_date,origin_airport,destination_airport,dep_delay,arr_delay,delay_total,row_number,dense_rank
AA,American Airlines,FL-125098,2025-03-31,ORD,DEN,73,81,154,3,2
AA,American Airlines,FL-148219,2025-06-16,SFO,ORD,70,79,149,4,3
AA,American Airlines,FL-128580,2025-03-24,DEN,SEA,74,71,145,5,4
AA,American Airlines,FL-128036,2025-03-30,ORD,LAS,70,72,142,6,5
B6,JetBlue Airways,FL-149603,2025-05-07,DFW,SEA,66,104,170,6,5
DL,Delta Air Lines,FL-148605,2025-04-21,JFK,DEN,66,79,145,6,5
NK,Spirit Airlines,FL-130965,2025-05-11,ORD,JFK,83,92,175,4,3
NK,Spirit Airlines,FL-141900,2025-06-17,DFW,JFK,83,91,174,5,4
NK,Spirit Airlines,FL-143441,2025-02-03,JFK,DFW,92,79,171,6,5
NK,Spirit Airlines,FL-132099,2025-04-20,SFO,DFW,76,95,171,7,5


#### 3.3 Ranking de aeropuertos por retraso promedio de vuelos salientes

`avg_delay` = retraso promedio de salida de los vuelos que parten del aeropuerto. `airport_rank = 1` es el aeropuerto con **más** retraso.

In [0]:
w_airport = Window.orderBy(F.col("avg_delay").desc())

airport_delay_ranking = (flights_s
    .groupBy("origin_airport", "origin_city", "origin_country")
    .agg(F.round(F.avg("dep_delay"), 2).alias("avg_delay"),
         F.round(F.avg("delay_total"), 2).alias("avg_delay_total"),
         F.count("*").alias("num_departures"))
    .withColumn("airport_rank", F.dense_rank().over(w_airport))
    .orderBy("airport_rank"))

display(airport_delay_ranking)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


origin_airport,origin_city,origin_country,avg_delay,avg_delay_total,num_departures,airport_rank
DFW,Dallas,USA,8.85,22.63,4915,1
LAS,Las Vegas,USA,8.82,22.73,4962,2
ORD,Chicago,USA,8.52,22.03,5007,3
SFO,San Francisco,USA,8.39,21.78,4929,4
LAX,Los Angeles,USA,8.38,21.74,4965,5
SEA,Seattle,USA,8.33,21.76,4891,6
MIA,Miami,USA,8.3,21.44,4878,7
ATL,Atlanta,USA,8.13,21.15,5010,8
JFK,New York,USA,8.11,21.35,4899,9
DEN,Denver,USA,8.07,20.96,5023,10


#### 3.4 Retraso promedio, % a tiempo y total de vuelos por aerolínea y mes

In [0]:
airline_month_metrics = (flights_time
    .groupBy("airline_id", "airline_name", "month", "month_name")
    .agg(F.round(F.avg("arr_delay"), 2).alias("avg_arr_delay"),
         F.round(F.avg("delay_total"), 2).alias("avg_delay_total"),
         F.round(F.avg(F.col("on_time").cast("int")) * 100, 2).alias("pct_on_time"),
         F.count("*").alias("total_flights"))
    .orderBy("airline_id", "month"))

display(airline_month_metrics)

airline_id,airline_name,month,month_name,avg_arr_delay,avg_delay_total,pct_on_time,total_flights
AA,American Airlines,1,January,10.8,16.43,57.83,1501
AA,American Airlines,2,February,9.63,14.75,60.56,1316
AA,American Airlines,3,March,8.6,12.75,61.78,1426
AA,American Airlines,4,April,9.56,14.29,59.33,1409
AA,American Airlines,5,May,10.07,15.2,59.91,1372
AA,American Airlines,6,June,10.46,15.65,60.41,1407
B6,JetBlue Airways,1,January,20.54,35.82,41.75,1363
B6,JetBlue Airways,2,February,19.31,33.74,42.59,1282
B6,JetBlue Airways,3,March,19.63,34.1,42.99,1440
B6,JetBlue Airways,4,April,19.82,34.43,40.3,1330


#### 3.5 Guardar resultados en Delta Lake y crear vistas temporales

In [0]:
resultados = {
    "delay_by_month": delay_by_month,
    "delay_by_weekday": delay_by_weekday,
    "delay_by_quarter": delay_by_quarter,
    "top5_delays_row_number": top5_row_number,
    "top5_delays_dense_rank": top5_dense_rank,
    "airport_delay_ranking": airport_delay_ranking,
    "airline_month_metrics": airline_month_metrics,
}
for nombre, df in resultados.items():
    guardar_delta(df, "analytics", nombre)
    df.createOrReplaceTempView(f"vw_{nombre}")

print("Vistas temporales:", [f"vw_{n}" for n in resultados])

Guardada aerolinea_trabajo_final.analytics.delay_by_month: 6 filas
Guardada aerolinea_trabajo_final.analytics.delay_by_weekday: 7 filas
Guardada aerolinea_trabajo_final.analytics.delay_by_quarter: 2 filas
Guardada aerolinea_trabajo_final.analytics.top5_delays_row_number: 30 filas
Guardada aerolinea_trabajo_final.analytics.top5_delays_dense_rank: 36 filas


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Guardada aerolinea_trabajo_final.analytics.airport_delay_ranking: 10 filas
Guardada aerolinea_trabajo_final.analytics.airline_month_metrics: 36 filas
Vistas temporales: ['vw_delay_by_month', 'vw_delay_by_weekday', 'vw_delay_by_quarter', 'vw_top5_delays_row_number', 'vw_top5_delays_dense_rank', 'vw_airport_delay_ranking', 'vw_airline_month_metrics']


In [0]:
%sql
-- Ejemplo de consulta para Databricks SQL
SELECT airline_name, month_name, avg_arr_delay, pct_on_time, total_flights
FROM aerolinea_trabajo_final.analytics.airline_month_metrics
ORDER BY pct_on_time DESC
LIMIT 10

airline_name,month_name,avg_arr_delay,pct_on_time,total_flights
American Airlines,March,8.6,61.78,1426
United Airlines,March,9.9,61.53,1375
United Airlines,April,10.25,61.08,1331
Delta Air Lines,January,9.96,61.01,1431
United Airlines,February,9.19,60.84,1305
Delta Air Lines,March,9.65,60.67,1363
United Airlines,May,10.38,60.65,1291
Delta Air Lines,February,9.72,60.64,1278
Southwest Airlines,May,9.41,60.57,1443
American Airlines,February,9.63,60.56,1316
